# Stage 2 — Data Preprocessing

This notebook turns the raw LinkedIn Job Postings 2023-2024 tables into one clean, salary-tiered table, `data/processed/cleaned_jobs.csv`, with these columns:

`job_id, job_title, company_name, location, experience_level, industry, skills_list, matched_skills, normalized_salary, salary_tier`

It also:
- cleans the 1.3M LinkedIn Jobs & Skills skill strings into per-job lists (`data/processed/linkedin_jobs_skills.csv`)
- builds a 100-skill vocabulary from those lists and matches it against each posting's description, producing `matched_skills`: real skills tied to salary data
- compares the salary tiers against the Data Science Salaries dataset
- writes a summary to `outputs/02_summary.txt`

The logic lives in `src/preprocessing.py`. This notebook calls it one step at a time so each intermediate result can be inspected. To run everything without the notebook: `python -m src.preprocessing`.

## Step 1 — Setup

Find the project root, add it to `sys.path` so `src` can be imported, and send the module's log messages to the notebook.

In [ ]:
import logging
import sys
from pathlib import Path

import pandas as pd


def find_root(start: Path) -> Path:
    """Return the first directory at or above `start` that contains CLAUDE.md."""
    for candidate in [start, *start.parents]:
        if (candidate / "CLAUDE.md").exists():
            return candidate
    raise FileNotFoundError("Could not locate project root (no CLAUDE.md found above cwd).")


PROJECT_ROOT = find_root(Path.cwd().resolve())
sys.path.insert(0, str(PROJECT_ROOT))

from src import preprocessing as pp

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s", force=True)

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

stats = {}

## Step 2 — Load postings and join the related tables

`postings.csv` is the main table, with one row per `job_id`. It gets three left joins:
- **`jobs/salaries.csv`**: fills any salary field that postings leaves empty. In the current data both files hold identical salary values, so this fills 0 values. It's kept as a consistency check.
- **`jobs/job_skills.csv` + `mappings/skills.csv`**: skill codes (e.g. `IT`) become names, collected into a list per job.
- **`jobs/job_industries.csv` + `mappings/industries.csv`**: industry names per job, joined with `"; "` (a job can have up to 3).

In [ ]:
df = pp.load_postings(RAW_DIR)
stats["postings_rows"] = len(df)
before = len(df)
df = df.drop_duplicates("job_id")
stats["dropped_duplicate_job_id"] = before - len(df)

df, stats["salary_values_from_salaries_csv"] = pp.attach_salaries(df, RAW_DIR)
df = pp.attach_skills(df, RAW_DIR)
df = pp.attach_industries(df, RAW_DIR)
df[["job_id", "title", "skills_list", "industry", "normalized_salary", "pay_period"]].head()

## Step 3 — Normalize titles and experience level

Titles are lowercased and whitespace is collapsed. A missing `formatted_experience_level` (about 24% of postings) becomes `"Unknown"` so it can still be grouped and one-hot encoded later.

In [ ]:
df = pp.normalize_titles(df)
df["experience_level"] = df["formatted_experience_level"].fillna("Unknown")
df["experience_level"].value_counts()

## Step 4 — Annualize salaries

`normalized_salary` is the main salary field. Where it's missing but a raw salary exists, it's computed as the midpoint of `min_salary`/`max_salary` (or `med_salary`) times the `pay_period` multiplier:

| pay_period | multiplier |
|---|---|
| HOURLY | 2080 |
| WEEKLY | 52 |
| BIWEEKLY | 26 |
| MONTHLY | 12 |
| YEARLY | 1 |

This matches how the dataset computes `normalized_salary` itself: on every row that has both, the formula reproduces it exactly. The dataset already fills `normalized_salary` for every posting with a salary, so this step currently fills 0 values. It's there in case a future data refresh leaves gaps.

In [ ]:
df, stats["salary_values_annualized"] = pp.annualize_salary(df)
df["pay_period"].value_counts(dropna=False)

## Step 5 — Filter salary rows

Rows are removed in this order:
1. **No salary.** Both `normalized_salary` and `min_salary` are null. This is about 71% of postings, because most LinkedIn postings don't list pay.
2. **Non-USD.** `normalized_salary` isn't converted between currencies, so mixing currencies would distort the tiers.
3. **Implausible values.** Annual salaries outside `[$10k, $1M]` are data-entry errors: $0, hourly rates tagged as yearly, and one $535M posting. Change the bounds with `pp.MIN_PLAUSIBLE_SALARY` / `pp.MAX_PLAUSIBLE_SALARY`.

In [ ]:
df = pp.filter_salary_rows(df, stats)
{k: stats[k] for k in ["dropped_no_salary", "dropped_non_usd", "dropped_implausible_salary"]}

### Step 5b — Drop reposted duplicates

The same ad is often posted under several `job_id`s, e.g. once per city, with identical company, title, description, and salary. Only one copy of each is kept. If the copies stayed in, they would land on both sides of Stage 5's train/test split, and the models would be scored partly on postings they had already seen. That inflated Random Forest's macro F1 by several points before this step was added.

In [ ]:
df = pp.drop_duplicate_postings(df, stats)
print(f"Dropped {stats['dropped_duplicate_postings']:,} duplicate postings -> {len(df):,} rows")

## Step 6 — Impute remaining missing salaries

Any salary still missing is filled with the median for its `experience_level`. A group with no known salaries falls back to the overall median. After Step 5 no salaries are missing, so this fills 0 values today.

In [ ]:
df, stats["salary_values_imputed"] = pp.impute_salary_by_experience(df)
df["normalized_salary"].describe()

## Step 7 — Discretize salary into tiers

The 33rd and 66th percentiles of `normalized_salary` split the jobs into three roughly equal groups: **Low** (bottom third), **Mid**, and **High** (top third). The cross-tab by experience level is a quick sanity check: Directors and Executives should be mostly High, and Interns mostly Low.

In [ ]:
df["salary_tier"], stats["tier_thresholds"] = pp.assign_salary_tier(df["normalized_salary"])
df["normalized_salary"] = df["normalized_salary"].round(2)

p33, p66 = stats["tier_thresholds"]
print(f"Low <= ${p33:,.0f} < Mid <= ${p66:,.0f} < High")
display(df["salary_tier"].value_counts())
(pd.crosstab(df["experience_level"], df["salary_tier"], normalize="index")
   .reindex(columns=pp.TIER_LABELS) * 100).round(1)

## Step 8 — Category skill lists

Each job's `skills_list` is a Python list of lowercased, de-duplicated skill names from `mappings/skills.csv`.

⚠️ **Limitation:** these are only **35 job-function categories** (e.g. *information technology*, *sales*, *engineering*), not concrete skills like *python* or *sql*. Jobs average under 2 of them. Steps 9-10 add concrete skills.

In [ ]:
skill_counts = df["skills_list"].explode().value_counts()
print(f"Distinct skills: {len(skill_counts)} | mean per job: {df['skills_list'].str.len().mean():.2f}")
skill_counts.head(15)

## Step 9 — Build a skill vocabulary from the 1.3M LinkedIn Jobs dataset

`linkedin_jobs/job_skills.csv` stores each job's skills as one comma-separated string, keyed by `job_link`. It can't be joined to the postings table because the two datasets share no key. Its skills are concrete, though, so its most common ones make a good vocabulary to search for in posting descriptions.

The raw names are messy: there are 2.77M distinct strings. Before the top `pp.TOP_N_SKILLS` (100) are taken:
- **Spelling variants are merged:** skills that differ only in spacing or punctuation (*problem solving* / *problemsolving* / *\* problem solving.*).
- **Synonyms are merged** (`pp.SKILL_SYNONYMS`): near-duplicates such as *communication skills* → *communication* and *microsoft excel* → *excel*, which would otherwise create trivial association rules.
- **Non-skills are excluded** (`pp.EXCLUDED_TERMS`): benefits and boilerplate such as *paid time off*, *401k*, *equal opportunity employer*, *vision*, and *medical*.

In [ ]:
jobs_skills = pp.load_linkedin_jobs_skills(RAW_DIR)
print(f"{len(jobs_skills):,} jobs, mean {jobs_skills['skills_list'].str.len().mean():.1f} skills per job")
vocab = pp.build_skill_vocabulary(jobs_skills)
vocab.head(20)

## Step 10 — Match vocabulary skills in posting descriptions

Each posting's `description` (plus `skills_desc`, when present) is searched for every vocabulary skill as a whole phrase. Words may be joined by spaces, hyphens, or nothing, so *problem-solving* matches *problem solving*.

**Nested phrases:** some skills contain another vocabulary skill (*project management* contains *management*, *written communication* contains *communication*). These longer phrases are matched first and then blanked out of the text. The shorter skill therefore counts only where it appears on its own. Otherwise every *project management* job would also get *management*, creating tautological association rules with confidence 1.0 in Stage 4.

The result, `matched_skills`, is the skill list Stages 4-5 should use.

This step takes about 2 minutes (100 regex scans over ~35k descriptions).

In [ ]:
df["matched_skills"] = pp.match_description_skills(df, vocab)
stats["final_rows"] = len(df)
cleaned = df[pp.OUTPUT_COLUMNS].reset_index(drop=True)

by_skill = (cleaned[["matched_skills", "normalized_salary"]].explode("matched_skills")
            .groupby("matched_skills")["normalized_salary"].agg(["count", "median"])
            .query("count >= 200").sort_values("median", ascending=False))
print("Highest and lowest median salary by matched skill (skills in >= 200 jobs):")
pd.concat([by_skill.head(8), by_skill.tail(8)])

## Step 11 — Salary benchmark against Data Science Salaries

The LinkedIn tier thresholds are compared with US full-time rows from the Data Science Salaries dataset. Data science pay is expected to sit mostly in the LinkedIn High tier, since the LinkedIn data covers all occupations.

In [ ]:
benchmark = pp.benchmark_ds_salaries(RAW_DIR, stats["tier_thresholds"], cleaned)
ds33, ds66 = benchmark["ds_thresholds"]
print(f"ds_salaries 33rd / 66th percentile: ${ds33:,.0f} / ${ds66:,.0f}")
print("Share of ds_salaries rows per LinkedIn tier:", benchmark["ds_share_in_linkedin_tiers"])
benchmark["median_by_experience"]

## Step 12 — Save outputs

- `data/processed/cleaned_jobs.csv`: the main cleaned table. `skills_list` is saved as a list literal. Read it back with `pp.load_cleaned_jobs(path)`, which parses it into Python lists.
- `data/processed/linkedin_jobs_skills.csv`: `job_link, skills_list` for about 1.3M jobs.
- `data/processed/skill_vocabulary.csv`: the 100 skills, their job counts, and the spellings searched for.
- `outputs/02_summary.txt`: the preprocessing summary.

In [ ]:
cleaned_path = PROCESSED_DIR / "cleaned_jobs.csv"
cleaned.to_csv(cleaned_path, index=False)
jobs_skills.to_csv(PROCESSED_DIR / "linkedin_jobs_skills.csv", index=False)
vocab.to_csv(PROCESSED_DIR / "skill_vocabulary.csv", index=False)

summary = pp.build_summary(cleaned, jobs_skills, vocab, stats, benchmark)
(OUTPUT_DIR / "02_summary.txt").write_text(summary, encoding="utf-8")

reloaded = pp.load_cleaned_jobs(cleaned_path)
assert list(reloaded.columns) == pp.OUTPUT_COLUMNS
assert isinstance(reloaded.loc[0, "skills_list"], list)
assert isinstance(reloaded.loc[0, "matched_skills"], list)
print(f"Saved {len(cleaned):,} rows to {cleaned_path}")

## Next steps

- Stage 3 loads `cleaned_jobs.csv` into the SQLite star schema.
- Stages 4 and 5 should use `matched_skills`: concrete skills tied to salary. Keep `skills_list` (35 categories) as an extra coarse feature.
- The vocabulary skews toward soft skills and credentials (*communication*, *high school diploma*), because the 1.3M dataset covers all occupations. Only a few technical skills (*python*, *sql*, *excel*) make the top 100. Raise `pp.TOP_N_SKILLS` for more technical coverage.